# DeepFM — Baseline vs Multimodal
**MovieLens 1M + TMDB (posters + sinopsis)**

Comparamos:
- **Modelo A (Baseline)**: DeepFM solo con IDs de usuario e ítem
- **Modelo B (Multimodal)**: DeepFM con IDs + géneros + embeddings de texto (sinopsis) + embeddings de imagen (poster)

Métrica principal: **NDCG@10** y **Precision@10**

## 0. Instalación

In [1]:
!pip install torch torchvision pandas numpy scikit-learn tqdm matplotlib kagglehub

## 1. Carga de datos

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.decomposition import PCA
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {DEVICE}")

# Rutas
import kagglehub, os
kaggle_path     = kagglehub.dataset_download("odedgolden/movielens-1m-dataset")
RATINGS_PATH    = f"{kaggle_path}/ratings.dat"
ITEM_TABLE_PATH = "item_table.csv"
TEXT_EMB_PATH   = "embeddings/text_embeddings.npy"
IMG_EMB_PATH    = "embeddings/image_embeddings.npy"

ratings = pd.read_csv(RATINGS_PATH, sep="::", engine="python",
                      names=["user_id", "movie_id", "rating", "timestamp"])
item_table = pd.read_csv(ITEM_TABLE_PATH)
text_emb   = np.load(TEXT_EMB_PATH)
img_emb    = np.load(IMG_EMB_PATH)

print(f"Ratings:          {ratings.shape}")
print(f"Items:            {item_table.shape}")
print(f"Text embeddings:  {text_emb.shape}")
print(f"Image embeddings: {img_emb.shape}")

Usando: cpu
Using Colab cache for faster access to the 'movielens-1m-dataset' dataset.
Ratings:          (1000209, 4)
Items:            (3883, 9)
Text embeddings:  (3883, 384)
Image embeddings: (3883, 768)


## 2. Split temporal 80/20 por usuario

In [4]:
ratings = ratings.sort_values(["user_id", "timestamp"])

def temporal_split(df, test_frac=0.2):
    train_idx, test_idx = [], []
    for _, grp in df.groupby("user_id"):
        n   = len(grp)
        cut = max(1, int(n * test_frac))
        train_idx.extend(grp.index[:-cut])
        test_idx.extend(grp.index[-cut:])
    return df.loc[train_idx].reset_index(drop=True), df.loc[test_idx].reset_index(drop=True)

train_df, test_df = temporal_split(ratings, test_frac=0.2)

# Solo positivos (rating >= 4)
train_pos = train_df[train_df["rating"] >= 4].copy()
test_pos  = test_df[test_df["rating"] >= 4].copy()

print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")
print(f"Train positivos: {len(train_pos):,} | Test positivos: {len(test_pos):,}")

Train: 802,553 | Test: 197,656
Train positivos: 472,129 | Test positivos: 103,152


## 3. Encoders y features multimodales

In [5]:
# Encoders de IDs
all_users  = sorted(ratings["user_id"].unique())
all_items  = sorted(ratings["movie_id"].unique())
user2idx   = {u: i for i, u in enumerate(all_users)}
item2idx   = {m: i for i, m in enumerate(all_items)}
n_users    = len(all_users)
n_items    = len(all_items)
print(f"Usuarios: {n_users} | Ítems: {n_items}")

# Géneros one-hot
item_table["genres_list"] = item_table["genres"].fillna("").apply(lambda g: g.split("|"))
mlb          = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(item_table["genres_list"]).astype(np.float32)
N_GENRES     = genre_matrix.shape[1]
print(f"Géneros: {N_GENRES}")

# PCA embeddings de texto
PCA_DIM      = 128
text_raw     = text_emb[item_table["emb_idx"].values]
text_pca     = PCA(n_components=PCA_DIM, random_state=42)
text_reduced = text_pca.fit_transform(text_raw).astype(np.float32)
print(f"Text PCA varianza: {text_pca.explained_variance_ratio_.sum():.1%}")

#  PCA embeddings de imagen (solo filas no vacías)
img_raw   = img_emb[item_table["emb_idx"].values]
zero_mask = (img_raw == 0).all(axis=1)
print(f"Películas sin poster: {zero_mask.sum()} ({zero_mask.mean():.1%})")
img_pca      = PCA(n_components=PCA_DIM, random_state=42)
img_reduced  = np.zeros((len(img_raw), PCA_DIM), dtype=np.float32)
img_reduced[~zero_mask] = img_pca.fit_transform(img_raw[~zero_mask])
print(f"Image PCA varianza: {img_pca.explained_variance_ratio_.sum():.1%}")

# Normalizar por bloque y concatenar
genre_norm   = normalize(genre_matrix,  norm="l2")
text_norm    = normalize(text_reduced,  norm="l2")
img_norm     = normalize(img_reduced,   norm="l2")
item_features = np.hstack([genre_norm, text_norm, img_norm]).astype(np.float32)
FEAT_DIM     = item_features.shape[1]
print(f"Feature matrix: {item_features.shape}")

# Mapear movie_id cn fila en item_features
movieid2feat_idx = {row["movie_id"]: i for i, row in item_table.iterrows()}

Usuarios: 6040 | Ítems: 3706
Géneros: 18
Text PCA varianza: 82.9%
Películas sin poster: 239 (6.2%)
Image PCA varianza: 68.2%
Feature matrix: (3883, 274)


## 4. Dataset PyTorch — muestreo negativo

Para cada interacción positiva generamos 4 negativos aleatorios (ítems no vistos por el usuario).

In [6]:
# set de items positivos por usuario
user_positives = train_pos.groupby("user_id")["movie_id"].apply(set).to_dict()

class InteractionDataset(Dataset):
    def __init__(self, pos_df, user2idx, item2idx, item_features,
                 movieid2feat_idx, all_items, n_neg=4, use_features=True):
        self.pairs         = list(zip(pos_df["user_id"], pos_df["movie_id"]))
        self.user2idx      = user2idx
        self.item2idx      = item2idx
        self.item_features = item_features
        self.mid2feat      = movieid2feat_idx
        self.all_items     = np.array(all_items)
        self.user_pos      = user_positives
        self.n_neg         = n_neg
        self.use_features  = use_features

    def __len__(self):
        return len(self.pairs) * (1 + self.n_neg)

    def __getitem__(self, idx):
        pos_idx   = idx // (1 + self.n_neg)
        neg_rank  = idx  % (1 + self.n_neg)
        user_id, pos_item = self.pairs[pos_idx]

        if neg_rank == 0:
            item_id, label = pos_item, 1.0
        else:
            seen = self.user_pos.get(user_id, set())
            while True:
                neg = np.random.choice(self.all_items)
                if neg not in seen:
                    item_id, label = neg, 0.0
                    break

        u_idx = self.user2idx[user_id]
        i_idx = self.item2idx.get(item_id, 0)

        if self.use_features:
            feat_idx = self.mid2feat.get(item_id, 0)
            feats    = torch.tensor(self.item_features[feat_idx], dtype=torch.float32)
        else:
            feats = torch.zeros(1)

        return (
            torch.tensor(u_idx, dtype=torch.long),
            torch.tensor(i_idx, dtype=torch.long),
            feats,
            torch.tensor(label, dtype=torch.float32)
        )

## 5. Arquitectura DeepFM

In [14]:
class DeepFM(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64,
                 feat_dim=0, use_features=True,
                 mlp_dims=(512, 256, 128), dropout=0.2):
        super().__init__()
        self.use_features = use_features
        self.emb_dim      = emb_dim

        # Embeddings de ID
        self.user_emb  = nn.Embedding(n_users, emb_dim, sparse=False)
        self.item_emb  = nn.Embedding(n_items, emb_dim, sparse=False)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        if use_features and feat_dim > 0:
            # Proyecciones separadas por modalidad
            n_genres   = 18
            n_text     = (feat_dim - n_genres) // 2
            n_img      = feat_dim - n_genres - n_text

            self.genre_proj = nn.Linear(n_genres, emb_dim)
            self.text_proj  = nn.Sequential(
                nn.Linear(n_text, emb_dim),
                nn.LayerNorm(emb_dim),
                nn.ReLU()
            )
            self.img_proj = nn.Sequential(
                nn.Linear(n_img, emb_dim),
                nn.LayerNorm(emb_dim),
                nn.ReLU()
            )
            # Peso aprendible por modalidad
            self.modal_attn = nn.Linear(emb_dim * 3, 3)

            mlp_input_dim = emb_dim * 3
        else:
            mlp_input_dim = emb_dim * 2

        # MLP
        layers, in_dim = [], mlp_input_dim
        for out_dim in mlp_dims:
            layers += [
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_ids, item_ids, feats=None):
        u_emb = self.user_emb(user_ids)
        i_emb = self.item_emb(item_ids)

        # FM
        fm_score  = (u_emb * i_emb).sum(dim=1, keepdim=True)
        fm_score += self.user_bias(user_ids) + self.item_bias(item_ids) + self.global_bias

        # Deep
        if self.use_features and feats is not None:
            n_genres = 18
            n_text   = (feats.shape[1] - n_genres) // 2

            g_emb = self.genre_proj(feats[:, :n_genres])
            t_emb = self.text_proj(feats[:, n_genres:n_genres + n_text])
            i_femb = self.img_proj(feats[:, n_genres + n_text:])

            modal_concat = torch.cat([g_emb, t_emb, i_femb], dim=1)
            attn_weights = torch.softmax(self.modal_attn(modal_concat), dim=1)
            feat_combined = (
                attn_weights[:, 0:1] * g_emb +
                attn_weights[:, 1:2] * t_emb +
                attn_weights[:, 2:3] * i_femb
            )

            deep_in = torch.cat([u_emb, i_emb, feat_combined], dim=1)
        else:
            deep_in = torch.cat([u_emb, i_emb], dim=1)

        deep_score = self.mlp(deep_in)
        return torch.sigmoid(fm_score + deep_score).squeeze(1)

## 6. Entrenamiento

In [17]:
BATCH_SIZE = 4096
EPOCHS     = 30
EMB_DIM    = 64

def train_model(model, train_loader, epochs=30, lr=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.BCELoss()
    model.to(DEVICE)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for u, i, feats, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            u, i, feats, labels = u.to(DEVICE), i.to(DEVICE), feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            preds = model(u, i, feats if model.use_features else None)
            loss  = criterion(preds, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        scheduler.step(avg_loss)
        print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    return model

# DataLoaders

ds_baseline = InteractionDataset(
    train_pos, user2idx, item2idx, item_features,
    movieid2feat_idx, all_items, n_neg=4, use_features=False
)
ds_multimodal = InteractionDataset(
    train_pos, user2idx, item2idx, item_features,
    movieid2feat_idx, all_items, n_neg=4, use_features=True
)

loader_baseline   = DataLoader(ds_baseline,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
loader_multimodal = DataLoader(ds_multimodal, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)



In [18]:
# Modelo A: Baseline
print("=" * 50)
print("Entrenando Modelo A — Baseline")
print("=" * 50)
model_baseline = DeepFM(
    n_users=n_users, n_items=n_items,
    emb_dim=EMB_DIM, feat_dim=0,
    use_features=False,
    mlp_dims=(256, 128, 64), dropout=0.3
)
model_baseline = train_model(model_baseline, loader_baseline, epochs=EPOCHS, lr=1e-3)



Entrenando Modelo A — Baseline


  Epoch   1 | Loss: 0.3507 | LR: 1.00e-03


  Epoch   2 | Loss: 0.3269 | LR: 1.00e-03


  Epoch   3 | Loss: 0.3047 | LR: 1.00e-03


  Epoch   4 | Loss: 0.2823 | LR: 1.00e-03


...

  Epoch  26 | Loss: 0.2038 | LR: 1.00e-03


  Epoch  27 | Loss: 0.2026 | LR: 1.00e-03


  Epoch  28 | Loss: 0.2015 | LR: 1.00e-03


  Epoch  29 | Loss: 0.2006 | LR: 1.00e-03


  Epoch  30 | Loss: 0.1994 | LR: 1.00e-03


In [20]:
# Modelo B: Multimodal
print("=" * 50)
print("Entrenando Modelo B — Multimodal")
print("=" * 50)
model_multimodal = DeepFM(
    n_users=n_users, n_items=n_items,
    emb_dim=EMB_DIM, feat_dim=FEAT_DIM,
    use_features=True,
    mlp_dims=(256, 128, 64), dropout=0.3
)
model_multimodal = train_model(model_multimodal, loader_multimodal, epochs=EPOCHS, lr=1e-3)

print("\n✓ Entrenamiento completado")

Entrenando Modelo B — Multimodal


  Epoch   1 | Loss: 0.3379 | LR: 1.00e-03


  Epoch   2 | Loss: 0.2817 | LR: 1.00e-03


  Epoch   3 | Loss: 0.2636 | LR: 1.00e-03


  Epoch   4 | Loss: 0.2539 | LR: 1.00e-03


...

  Epoch  26 | Loss: 0.2002 | LR: 1.00e-03


  Epoch  27 | Loss: 0.1995 | LR: 1.00e-03


  Epoch  28 | Loss: 0.1980 | LR: 1.00e-03


  Epoch  29 | Loss: 0.1972 | LR: 1.00e-03


  Epoch  30 | Loss: 0.1959 | LR: 1.00e-03

✓ Entrenamiento completado


## 7. Evaluación — Precision@10 y NDCG@10

In [21]:
def get_top10_deepfm(model, user_id, all_items, user2idx, item2idx,
                     item_features, movieid2feat_idx, k=10):
    model.eval()
    u_idx  = user2idx.get(user_id)
    if u_idx is None:
        return []

    u_tensor = torch.tensor([u_idx] * len(all_items), dtype=torch.long).to(DEVICE)
    i_tensor = torch.tensor([item2idx.get(m, 0) for m in all_items], dtype=torch.long).to(DEVICE)

    if model.use_features:
        feat_idx = [movieid2feat_idx.get(m, 0) for m in all_items]
        f_tensor = torch.tensor(item_features[feat_idx], dtype=torch.float32).to(DEVICE)
    else:
        f_tensor = None

    with torch.no_grad():
        scores = []
        chunk  = 512
        for start in range(0, len(all_items), chunk):
            end = start + chunk
            f_chunk = f_tensor[start:end] if f_tensor is not None else None
            s = model(u_tensor[start:end], i_tensor[start:end], f_chunk)
            scores.append(s.cpu().numpy())
        scores = np.concatenate(scores)

    top_idx = np.argsort(-scores)[:k]
    return [all_items[i] for i in top_idx]


def evaluate_deepfm(model, test_pos, all_users, all_items,
                    user2idx, item2idx, item_features, movieid2feat_idx,
                    k=10, sample_users=500):
    """Evalúa sobre una muestra de usuarios con positivos en test."""
    test_ground_truth = test_pos.groupby("user_id")["movie_id"].apply(set).to_dict()
    eval_users = [u for u in test_ground_truth if len(test_ground_truth[u]) > 0]
    eval_users = np.random.choice(eval_users, min(sample_users, len(eval_users)), replace=False)

    precisions, ndcgs = [], []
    for user_id in tqdm(eval_users, desc="Evaluando"):
        relevant = test_ground_truth[user_id]
        top_k    = get_top10_deepfm(model, user_id, all_items, user2idx, item2idx,
                                    item_features, movieid2feat_idx, k=k)

        # Precision@K
        hits = sum(1 for m in top_k if m in relevant)
        precisions.append(hits / k)

        # NDCG@K
        dcg  = sum(1.0 / np.log2(r + 2) for r, m in enumerate(top_k) if m in relevant)
        idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(relevant), k)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return np.mean(precisions), np.mean(ndcgs)


print("Evaluando Baseline...")
prec_base, ndcg_base = evaluate_deepfm(
    model_baseline, test_pos, all_users, all_items,
    user2idx, item2idx, item_features, movieid2feat_idx
)

print("Evaluando Multimodal...")
prec_mm, ndcg_mm = evaluate_deepfm(
    model_multimodal, test_pos, all_users, all_items,
    user2idx, item2idx, item_features, movieid2feat_idx
)

results = pd.DataFrame({
    "Modelo":       ["Baseline (solo IDs)", "Multimodal (IDs + texto + imagen)"],
    "Precision@10": [prec_base, prec_mm],
    "NDCG@10":      [ndcg_base, ndcg_mm],
})
print("\n" + results.to_string(index=False, float_format="{:.4f}".format))

delta_prec = (prec_mm - prec_base) / prec_base * 100
delta_ndcg = (ndcg_mm - ndcg_base) / ndcg_base * 100
print(f"\nMejora relativa Multimodal vs Baseline:")
print(f"  Precision@10: {delta_prec:+.1f}%")
print(f"  NDCG@10:      {delta_ndcg:+.1f}%")

Evaluando Baseline...


Evaluando: 100%|██████████| 500/500 [00:09<00:00, 50.11it/s]


Evaluando Multimodal...


Evaluando: 100%|██████████| 500/500 [00:16<00:00, 29.60it/s]


                           Modelo  Precision@10  NDCG@10
              Baseline (solo IDs)        0.0250   0.0278
Multimodal (IDs + texto + imagen)        0.0266   0.0342

Mejora relativa Multimodal vs Baseline:
  Precision@10: +6.4%
  NDCG@10:      +23.0%
